# Lab 8: Capstone Preparation — Small-World Structure, Null Models, and Communities

This final lab is the logical next step from the previous module on hubs, centrality, and preferential attachment.

In that lab, you learned how to describe important vertices and compare network growth models.
In this lab, you will learn how to do something broader:

- describe global and mesoscale structure,
- compare an observed network to a null model,
- estimate empirical p-values,
- detect communities,
- measure modularity, homophily, and assortativity,
- organize those results into a capstone-ready workflow.

By the end of this lab, you should have the final toolkit needed to analyze a network for your capstone project.

In [ ]:
from scratch.old_labs.module_12.lab8_capstone_helpers import *

### Part 1: Warm-up — path length and clustering in familiar graphs

Before analyzing a real network, compare a few familiar graph families.

In [ ]:
P = path_graph(12)
C = cycle_graph(12)
S = star_graph(12)
R = random_graph(12, 14, seed=10)

In [ ]:
graph_metrics(P), graph_metrics(C), graph_metrics(S), graph_metrics(R)

In [ ]:
draw_graph(P, "Path graph", with_labels=False)
draw_graph(C, "Cycle graph", with_labels=False)
draw_graph(S, "Star graph", with_labels=False)
draw_graph(R, "Random graph", with_labels=False)

Questions:
- Which graph has the largest average path length?
- Which graph has the smallest average path length?
- Which graph has the highest clustering?
- Which graph feels most small-world-like at first glance?

### Part 2: Observed network and descriptive statistics

Use Zachary's Karate Club graph as an observed network and compute its main summary statistics.

In [ ]:
K = karate_graph()
graph_summary(K)

In [ ]:
draw_graph(K, "Observed graph: Zachary's Karate Club", with_labels=True)

In [ ]:
observed_metrics = graph_metrics(K)
observed_metrics

Questions:
- Is the graph connected?
- Does it appear visually clustered?
- Which statistics from this section would help support or challenge a small-world claim?

### Part 3: State a null hypothesis and simulate a null distribution

We test whether the observed clustering is unusually large relative to an Erdős–Rényi null model with the same number of vertices and edges.

\[
H_0: \text{The observed graph is consistent with a } G(n,m) \text{ random graph model.}
\]

\[
H_1: \text{The observed graph has larger average clustering than expected under } G(n,m).
\]

In [ ]:
null_df = simulate_matching_observed(K, trials=300, seed=12)
null_df.head()

In [ ]:
observed_clustering = observed_metrics["average_clustering"]
observed_path_length = observed_metrics["average_path_length"]
observed_clustering, observed_path_length

### Part 4: Compare the observed graph to the null distribution

Now compare the observed clustering and average path length to the simulated null distribution.

In [ ]:
plot_metric_histogram(
    null_df["average_clustering"],
    observed_value=observed_clustering,
    xlabel="Average clustering",
    title="Null distribution of average clustering under G(n,m)"
)

In [ ]:
empirical_p_value_upper(null_df["average_clustering"], observed_clustering)

In [ ]:
plot_metric_histogram(
    null_df["average_path_length"],
    observed_value=observed_path_length,
    xlabel="Average path length",
    title="Null distribution of average path length under G(n,m)"
)

In [ ]:
empirical_p_value_two_sided(null_df["average_path_length"], observed_path_length)

Questions:
- Does the observed clustering appear unusual under the null?
- Does the observed path length appear unusual under the null?
- Which statistic gives stronger evidence against the baseline model?

### Part 5: Why repeated simulation matters

A single comparison graph is not enough for statistical testing.

In [ ]:
single_random = er_graph_matching_observed(K, seed=111)
graph_metrics(single_random)

In [ ]:
draw_graph(single_random, "One null-model comparison graph", with_labels=False)

Questions:
- Why is one simulated graph not enough?
- What does the null distribution tell you that one graph cannot?
- How does this change the way you interpret an observed network statistic?

### Part 6: Community detection and modularity

Now shift from global and local structure to mesoscale structure.

We ask whether the graph appears to break into communities.

In [ ]:
communities_K = detect_communities_greedy(K)
communities_K

In [ ]:
community_size_table(communities_K)

In [ ]:
modularity_score(K, communities_K)

In [ ]:
draw_graph_colored_by_partition(K, communities_K, "Karate Club graph colored by detected community", with_labels=True)

Questions:
- How many communities were detected?
- Does the partition look visually plausible?
- What does the modularity score suggest?
- Why is a detected partition not automatically proof of a meaningful social division?

### Part 7: Compare community structure to a random graph

Even a random graph can produce apparent clusters and nontrivial modularity.

In [ ]:
R2 = random_graph(K.number_of_nodes(), K.number_of_edges(), seed=20)
graph_summary(R2)

In [ ]:
communities_R2 = detect_communities_greedy(R2)
community_size_table(communities_R2)

In [ ]:
modularity_score(R2, communities_R2)

In [ ]:
draw_graph_colored_by_partition(R2, communities_R2, "Random graph colored by detected community", with_labels=False)

Questions:
- Does the random graph also produce a partition?
- Is that surprising?
- Why is ``an algorithm found communities'' not the same as ``the network has meaningful communities''?

### Part 8: Strong versus weak community structure

Compare two graphs that both have two planted groups, but with different structural strength.

In [ ]:
P_strong = planted_two_group_graph()
P_weak = weak_two_group_graph()

In [ ]:
communities_strong = detect_communities_greedy(P_strong)
communities_weak = detect_communities_greedy(P_weak)

In [ ]:
modularity_score(P_strong, communities_strong), modularity_score(P_weak, communities_weak)

In [ ]:
draw_graph_colored_by_partition(P_strong, communities_strong, "Strong two-group graph", with_labels=True)
draw_graph_colored_by_partition(P_weak, communities_weak, "Weak two-group graph", with_labels=True)

Questions:
- Which graph has stronger community structure?
- How does modularity reflect that difference?
- What kinds of edges weaken a community partition?

### Part 9: Homophily and assortativity

Now study a graph with labeled node types.

Homophily is a tendency for similar vertices to connect to one another.
Assortativity gives a numerical way to measure that tendency.

In [ ]:
H = labeled_homophily_graph()
graph_summary(H)

In [ ]:
draw_graph(H, "Labeled homophily graph", with_labels=True, color_attribute="color_group")

In [ ]:
attribute_mixing_table(H, "color_group")

In [ ]:
attribute_assortativity(H, "color_group")

In [ ]:
degree_assortativity(H)

Questions:
- Do red nodes tend to connect to red nodes?
- Do blue nodes tend to connect to blue nodes?
- How is assortativity by node label different from degree assortativity?
- Why might homophily help explain observed community structure?

### Part 10: A capstone-ready workflow summary

Now summarize an observed graph the way you might for your final project.

In [ ]:
capstone_workflow_summary(K, trials=200, seed=30)

This summary gives you a template for a capstone analysis:

- compute descriptive graph statistics,
- compare a chosen statistic to a null distribution,
- estimate an empirical p-value,
- detect communities,
- report modularity.

This is not the only possible workflow, but it is a strong baseline.

### Part 11: Gephi exploration

Export the key graphs to Gephi and compare what the visualizations suggest.

In [ ]:
export_for_gephi(K, "lab08_karate_observed.gexf")
export_for_gephi(single_random, "lab08_single_null_graph.gexf")
export_for_gephi(P_strong, "lab08_planted_two_group.gexf")
export_for_gephi(H, "lab08_homophily_graph.gexf")

In Gephi:

1. Import the observed graph and the null-model comparison graph.
2. Apply the same layout to both.
3. Compare clustering, grouping, and visual ``clumpiness.''
4. Import the planted and homophily graphs and compare community or attribute-based structure.

Questions:
- Which graph shows the clearest local clustering?
- Which graph shows the clearest community structure?
- Which graph shows homophily most visibly?
- What can the picture suggest, and what still requires a statistic or a test?

### Part 12: Reflection and capstone planning

Answer the following in complete sentences.

1. What does average path length measure?
2. What does clustering measure?
3. What does an empirical p-value mean in this setting?
4. Why is repeated simulation necessary?
5. What does modularity measure?
6. Why can a random graph still produce a detected partition?
7. What is homophily?
8. What does assortativity measure?
9. Which two or three tools from this lab do you think will be most useful in your capstone project?
10. What network question could you now investigate that you could not have investigated before this module?